# Preprocessing

In this notebook, a series of preprocessing steps are performed on the data. 
Note that some datasets may not require all of these steps, and some could include additional steps that are not listed here and could interfere with the performance of the segmentation tools.

## DIY: Data preparation
The data should be organized in a specific way to be used by the segmentation tools. Otherwise, in order to train models/test the existing ones on new datasets, you should define a new datamodel (see src/datamodels.py).
The following is an example of how the data should be organized:
```
wmh
├── training
│   ├── center_name
│   │   ├── subject_id
│   │   │   ├── pre
│   │   │   │   ├── FLAIR.nii.gz
│   │   │   │   ├── T1.nii.gz
│   │   │   ├── wmh.nii.gz
├── test
│   ├── center_name
│   │   ├── subject_id
│   │   │   ├── pre
│   │   │   │   ├── FLAIR.nii.gz
│   │   │   │   ├── T1.nii.gz
```

Note that even the images are located in a 'pre' folder, they are not preprocessed yet, but they will be in the following steps.

# Bias field correction
The pretrained models were trained on images that were bias-field corrected.

In [21]:
import os

import SimpleITK as sitk

src_folder = os.path.expanduser('~/Code/datasets/wmh/training/UMCL')
dst_folder = os.path.expanduser('~/Code/datasets/wmh_prep/training/UMCL')

filename_filter = ['FLAIR.nii.gz', 'T1.nii.gz']
move_also = ['wmh.nii.gz']

# run the bias field correction preserving the folder structure for all subfolders
if not os.path.exists(src_folder):
    raise ValueError(f'Folder {src_folder} does not exist')
for root, dirs, files in os.walk(src_folder):
    out_folder = os.path.join(dst_folder, os.path.relpath(root, src_folder))
    os.makedirs(out_folder, exist_ok=True)
    for filename in files:
        if filename in filename_filter:
            print(f'Processing {os.path.join(root, filename)}')
            img = sitk.ReadImage(os.path.join(root, filename))
            corr = sitk.N4BiasFieldCorrection(sitk.Cast(img, sitk.sitkFloat32))
            sitk.WriteImage(corr, os.path.join(out_folder, filename))
        elif filename in move_also:
            print(f'Copying {os.path.join(root, filename)}')
            os.system(
                f'cp {os.path.join(root, filename)} {os.path.join(out_folder, filename)}')

Copying /home/fmatzkin/Code/datasets/wmh/training/UMCL/20/wmh.nii.gz
Processing /home/fmatzkin/Code/datasets/wmh/training/UMCL/20/pre/FLAIR.nii.gz
Processing /home/fmatzkin/Code/datasets/wmh/training/UMCL/20/pre/T1.nii.gz
Copying /home/fmatzkin/Code/datasets/wmh/training/UMCL/01/wmh.nii.gz
Processing /home/fmatzkin/Code/datasets/wmh/training/UMCL/01/pre/FLAIR.nii.gz
Processing /home/fmatzkin/Code/datasets/wmh/training/UMCL/01/pre/T1.nii.gz
Copying /home/fmatzkin/Code/datasets/wmh/training/UMCL/21/wmh.nii.gz
Processing /home/fmatzkin/Code/datasets/wmh/training/UMCL/21/pre/FLAIR.nii.gz
Processing /home/fmatzkin/Code/datasets/wmh/training/UMCL/21/pre/T1.nii.gz
Copying /home/fmatzkin/Code/datasets/wmh/training/UMCL/28/wmh.nii.gz
Processing /home/fmatzkin/Code/datasets/wmh/training/UMCL/28/pre/FLAIR.nii.gz
Processing /home/fmatzkin/Code/datasets/wmh/training/UMCL/28/pre/T1.nii.gz
Copying /home/fmatzkin/Code/datasets/wmh/training/UMCL/24/wmh.nii.gz
Processing /home/fmatzkin/Code/datasets/wmh

In [22]:
import SimpleITK as sitk
inp = '/home/fmatzkin/Code/datasets/wmh/training/Utrecht/0/orig/T1.nii.gz'
outp = '/home/fmatzkin/Code/datasets/wmh/training/Utrecht/0/orig/T1_bc.nii.gz'
img = sitk.ReadImage(inp)
corr = sitk.N4BiasFieldCorrection(sitk.Cast(img, sitk.sitkFloat32))
sitk.WriteImage(corr, outp)

# Take images to the same space
The pretrained models were trained on patches of images with the same spacing and orientation.
We will avoid registration, but since the image shapes vary across datasets, we will resample them after removing tissues far from the brain.

## Crop according to brain mask


In [14]:
margin = 5

from scipy.ndimage import binary_dilation
import os


def get_b_mask_path(subj_path):
    """ Load brain mask

    Given a subject path, check if the brain mask exists. If it does, return its
    path. If it doesn't, create it (using FSLs BET) and return its path.

    :param subj_path: Path to subject folder
    :return: Brain mask path
    """
    b_mask_path = os.path.join(subj_path, 'T1_brain_mask.nii.gz')
    if os.path.exists(b_mask_path):
        return b_mask_path
    else:
        t1_path = os.path.join(subj_path, 'T1.nii.gz')
        b_t1b_path = os.path.join(subj_path, 'T1_brain.nii.gz')
        cmd = f"bet {t1_path} {b_t1b_path} -m"
        os.system(cmd)
        print(f"Created brain mask for {subj_path}")
        return b_mask_path


def crop_image_to_mask_with_margin(image_path, mask_path, crop_files=[], 
                                   margin=0, n_dilate=0, out_file_id=None):
    """
    Crop an image to the region of interest (ROI) defined by a binary mask plus a custom margin using NumPy.

    Args:
        image_path (str): Path to the input image (NIfTI format).
        mask_path (str): Path to the binary mask image (NIfTI format).
        crop_files (list): List of paths to images to crop in the same way as the input image (default is []).
        margin (int): Custom margin size to add around the mask ROI (default is 0).
        n_dilate (int): Number of times to dilate the mask before cropping (default is 0).
        out_file_id (str): Custom suffix to add to the output file name (default is None -overwrite the input files-).
    
    Returns:
        None
    """
    # Load the input image and binary mask using nibabel
    image = nib.load(image_path)
    mask = nib.load(mask_path)

    # Get data arrays from the images
    image_data = image.get_fdata()
    mask_data = mask.get_fdata()

    # Dilate mask_data five times using a 3x3x3 structuring element
    # This is to avoid cropping too close to the brain
    if n_dilate > 0:
        mask_data = binary_dilation(mask_data, iterations=n_dilate,
                                    structure=np.ones((3, 3, 3)))

    # Find the indices of non-zero elements in the mask
    mask_indices = np.argwhere(mask_data > 0)

    # Determine the bounding box for the ROI
    min_indices = np.maximum(np.min(mask_indices, axis=0) - margin, 0)
    max_indices = np.minimum(np.max(mask_indices, axis=0) + margin + 1,
                             mask_data.shape)

    # Crop the images
    for img_path in crop_files:
        img = nib.load(img_path)
        img_data = img.get_fdata()
        cropped_img_data = img_data[
                           min_indices[0]:max_indices[0],
                           min_indices[1]:max_indices[1],
                           min_indices[2]:max_indices[2]
                           ]
        cropped_img = nib.Nifti1Image(cropped_img_data, affine=img.affine)
        out_path = img_path.replace('.nii.gz', f'_{out_file_id}.nii.gz') \
            if out_file_id is not None else img_path
        nib.save(cropped_img, out_path)


src_folder = os.path.expanduser('~/fast_dataset_cache/wmh_UtrechtSpace/wmh/training/Amsterdam')
dst_folder = os.path.expanduser('~/fast_dataset_cache/wmh_UtrechtSpace/wmh/training/Amsterdam')

for root, dirs, files in os.walk(src_folder):
    if 'pre' not in root:
        continue
    for filename in files:
        if filename == 'T1.nii.gz':
            t1_path = os.path.join(root, filename)
            flair_path = os.path.join(root, 'FLAIR.nii.gz')
            wmh_path = os.path.join(os.path.dirname(root), 'wmh.nii.gz')

            b_mask_path = get_b_mask_path(root)

            crop_files = [t1_path, flair_path, wmh_path, b_mask_path]
            crop_image_to_mask_with_margin(t1_path, b_mask_path, crop_files,
                                           margin)

            print(f'Processed {t1_path}')

Processed /home/fmatzkin/fast_dataset_cache/wmh_UtrechtSpace/wmh/training/Amsterdam/GE3T/103/pre/T1.nii.gz
Processed /home/fmatzkin/fast_dataset_cache/wmh_UtrechtSpace/wmh/training/Amsterdam/GE3T/132/pre/T1.nii.gz
Processed /home/fmatzkin/fast_dataset_cache/wmh_UtrechtSpace/wmh/training/Amsterdam/GE3T/109/pre/T1.nii.gz
Processed /home/fmatzkin/fast_dataset_cache/wmh_UtrechtSpace/wmh/training/Amsterdam/GE3T/116/pre/T1.nii.gz
Processed /home/fmatzkin/fast_dataset_cache/wmh_UtrechtSpace/wmh/training/Amsterdam/GE3T/115/pre/T1.nii.gz
Processed /home/fmatzkin/fast_dataset_cache/wmh_UtrechtSpace/wmh/training/Amsterdam/GE3T/107/pre/T1.nii.gz
Processed /home/fmatzkin/fast_dataset_cache/wmh_UtrechtSpace/wmh/training/Amsterdam/GE3T/126/pre/T1.nii.gz
Processed /home/fmatzkin/fast_dataset_cache/wmh_UtrechtSpace/wmh/training/Amsterdam/GE3T/110/pre/T1.nii.gz
Processed /home/fmatzkin/fast_dataset_cache/wmh_UtrechtSpace/wmh/training/Amsterdam/GE3T/144/pre/T1.nii.gz
Processed /home/fmatzkin/fast_dataset

## Resample according to reference image

We will use a reference image to resample all the images to the same space.

In [21]:

import numpy as np
import nibabel as nib
from nibabel.processing import conform
from nilearn.image import resample_to_img

src_folder = os.path.expanduser('~/fast_dataset_cache/wmh_UtrechtSpace/wmh/training/UMCL')
dst_folder = os.path.expanduser('~/fast_dataset_cache/wmh_UtrechtSpace/wmh/training/UMCL')
reference_img_path = os.path.expanduser('~/Code/datasets/wmh/training/Utrecht/0/pre/T1.nii.gz')

filename_filter = ['FLAIR.nii.gz', 'T1.nii.gz', 'wmh.nii.gz']

for root, dirs, files in os.walk(src_folder):
    for filename in files:
        if filename in filename_filter:
            t1_path = os.path.join(root, filename)
            base = nib.load(reference_img_path)
            img = nib.load(t1_path)
            interp = 'nearest' if filename == 'wmh.nii.gz' else 'continuous'
            # image = resample_to_img(img, base, interp)  # TODO FIX THIS
            image = conform(img, base.shape, base.header.get_zooms())
            nib.save(image, t1_path)
            print(f'Processed {t1_path}')
            

Processed /home/fmatzkin/fast_dataset_cache/wmh_UtrechtSpace/wmh/training/UMCL/22/wmh.nii.gz
Processed /home/fmatzkin/fast_dataset_cache/wmh_UtrechtSpace/wmh/training/UMCL/22/pre/T1.nii.gz
Processed /home/fmatzkin/fast_dataset_cache/wmh_UtrechtSpace/wmh/training/UMCL/22/pre/FLAIR.nii.gz
Processed /home/fmatzkin/fast_dataset_cache/wmh_UtrechtSpace/wmh/training/UMCL/06/wmh.nii.gz
Processed /home/fmatzkin/fast_dataset_cache/wmh_UtrechtSpace/wmh/training/UMCL/06/pre/T1.nii.gz
Processed /home/fmatzkin/fast_dataset_cache/wmh_UtrechtSpace/wmh/training/UMCL/06/pre/FLAIR.nii.gz
Processed /home/fmatzkin/fast_dataset_cache/wmh_UtrechtSpace/wmh/training/UMCL/05/wmh.nii.gz
Processed /home/fmatzkin/fast_dataset_cache/wmh_UtrechtSpace/wmh/training/UMCL/05/pre/T1.nii.gz
Processed /home/fmatzkin/fast_dataset_cache/wmh_UtrechtSpace/wmh/training/UMCL/05/pre/FLAIR.nii.gz
Processed /home/fmatzkin/fast_dataset_cache/wmh_UtrechtSpace/wmh/training/UMCL/18/wmh.nii.gz
Processed /home/fmatzkin/fast_dataset_cache